Arrays give you O(1) access by *position*. Hash maps give you O(1) access by *key* — and the key can be anything hashable: a string, a number, a tuple. This is the most powerful lookup structure in everyday programming. Mastering when and how to use hash maps is one of the fastest ways to improve your problem-solving speed — a huge number of interview problems that look like O(n²) reduce to O(n) the moment you reach for a hash map.

## How Hashing Works

Under the hood, a hash map is an array of **buckets**. To store a key-value pair:

1. Pass the key through a **hash function** — a deterministic function that converts the key into an integer
2. Take that integer modulo the array size to get a **bucket index**
3. Store the key-value pair in that bucket

To retrieve a value:
1. Hash the key → compute the same bucket index (same arithmetic, same result)
2. Go directly to that bucket — O(1), no searching

```
index = hash(key) % table_size
```

![Hash Map Internals](https://raw.githubusercontent.com/schemabotview/dsa/main/img/hashmap-internals.svg)

## Hash Collisions

Two different keys can produce the same bucket index — this is called a **collision**. The most common resolution strategy is **separate chaining**: each bucket holds a linked list of all key-value pairs that mapped to it. On lookup, you hash to the bucket and then scan the (usually tiny) list for your key.

Collisions are the reason hash map operations are **O(1) average** but **O(n) worst case**. In the pathological case where every key hashes to the same bucket, you degrade to a linked list scan. In practice, a good hash function with a reasonable **load factor** (the ratio of stored entries to bucket count) keeps chains very short — typically length 1.

**Load factor** controls when the table resizes. Python's `dict` resizes when the load factor exceeds about two-thirds, roughly doubling the table size and rehashing all keys. This keeps average chain length close to 1 and preserves O(1) performance.

## Complexity

| Operation | Average | Worst case | Why worst |
|---|---|---|---|
| `get(key)` | O(1) | O(n) | All keys in one bucket |
| `put(key, val)` | O(1) | O(n) | Same |
| `delete(key)` | O(1) | O(n) | Same |
| `contains(key)` | O(1) | O(n) | Same |
| Iteration | O(n) | O(n) | Must visit all entries |

In practice, with Python's `dict` or Kotlin's `HashMap`, you can treat all operations as O(1) — the worst case does not occur with normal data and a good hash function.

## Hash Map Operations

### Python

In [ ]:
# Creating and populating
person = {"name": "Alice", "age": 30, "city": "NYC"}

# Access — O(1)
print(person["name"])              # Alice
print(person.get("age"))           # 30
print(person.get("score", 0))      # 0  — default if key absent (no KeyError)

# Insert / update — O(1)
person["score"] = 95
person["age"]   = 31               # overwrites existing

# Delete — O(1)
del person["city"]
removed = person.pop("score")      # returns the removed value

# Membership — O(1)
print("name" in person)            # True
print("city" in person)            # False

# Iteration
for key, val in person.items():    # O(n)
    print(f"{key}: {val}")

print(list(person.keys()))         # ['name', 'age']
print(list(person.values()))       # ['Alice', 31]

### Kotlin

```kotlin
fun main() {
    // Immutable map
    val person = mapOf("name" to "Alice", "age" to 30, "city" to "NYC")
    println(person["name"])                  // Alice
    println(person.getOrDefault("score", 0)) // 0

    // Mutable map
    val scores = mutableMapOf("Alice" to 90, "Bob" to 85)
    scores["Carol"] = 92                     // insert
    scores["Alice"] = 95                     // update
    scores.remove("Bob")                     // delete

    println("Alice" in scores)              // true
    println("Bob" in scores)               // false

    for ((key, value) in scores) println("$key: $value")
}
```

## Useful Map Helpers

### Python

In [ ]:
from collections import defaultdict, Counter

# defaultdict — never raises KeyError; missing keys get a default value
graph = defaultdict(list)
graph[1].append(2)    # no need to initialise graph[1] = [] first
graph[1].append(3)
graph[2].append(4)
print(dict(graph))    # {1: [2, 3], 2: [4]}

# Counter — frequency map, built for counting
words  = ["apple", "banana", "apple", "cherry", "banana", "apple"]
freq   = Counter(words)
print(freq)                      # Counter({'apple': 3, 'banana': 2, 'cherry': 1})
print(freq.most_common(2))       # [('apple', 3), ('banana', 2)]
print(freq["apple"])             # 3
print(freq["mango"])             # 0  — missing keys return 0, not KeyError

# setdefault — insert default only if key is absent
d = {}
d.setdefault("x", []).append(1)
d.setdefault("x", []).append(2)  # key exists — default ignored
print(d)                          # {'x': [1, 2]}

### Kotlin

```kotlin
fun main() {
    // getOrPut — insert default only if key is absent (equivalent to setdefault)
    val graph = mutableMapOf<Int, MutableList<Int>>()
    graph.getOrPut(1) { mutableListOf() }.add(2)
    graph.getOrPut(1) { mutableListOf() }.add(3)
    println(graph)   // {1=[2, 3]}

    // Frequency count
    val words = listOf("apple", "banana", "apple", "cherry", "banana", "apple")
    val freq  = mutableMapOf<String, Int>()
    for (w in words) freq[w] = freq.getOrDefault(w, 0) + 1
    println(freq)    // {apple=3, banana=2, cherry=1}
}
```

## Hash Set

A **hash set** is a hash map with no values — only keys. It answers one question in O(1): *is this element in the set?*

Sets support the full mathematical vocabulary: union, intersection, difference, and subset checks.

### Python

In [ ]:
a = {1, 2, 3, 4, 5}
b = {3, 4, 5, 6, 7}

# Membership — O(1)
print(3 in a)              # True
print(9 in a)              # False

# Mutating — O(1)
a.add(6)
a.discard(1)               # remove if present, no error if absent

# Set operations — O(n)
print(a | b)               # union:        {2, 3, 4, 5, 6, 7}
print(a & b)               # intersection: {3, 4, 5, 6}
print(a - b)               # difference:   {2}
print(a ^ b)               # symmetric diff: {2, 7}
print({3, 4} <= a)         # subset check: True

### Kotlin

```kotlin
fun main() {
    val a = mutableSetOf(1, 2, 3, 4, 5)
    val b = setOf(3, 4, 5, 6, 7)

    println(3 in a)                  // true  — O(1)
    a.add(6)
    a.remove(1)

    println(a union b)               // [2, 3, 4, 5, 6, 7]
    println(a intersect b)           // [3, 4, 5, 6]
    println(a subtract b)            // [2]
    println(a.containsAll(setOf(3, 4)))  // true — subset check
}
```

## Pattern 1 — Frequency Counting

### Python

In [ ]:
from collections import Counter

# Is s2 an anagram of s1? — O(n)
def is_anagram(s1: str, s2: str) -> bool:
    return Counter(s1) == Counter(s2)

print(is_anagram("listen", "silent"))   # True
print(is_anagram("hello", "world"))     # False

# Top-k frequent elements — O(n log k)
def top_k(nums: list[int], k: int) -> list[int]:
    freq = Counter(nums)
    return [x for x, _ in freq.most_common(k)]

print(top_k([1, 1, 1, 2, 2, 3], k=2))  # [1, 2]

### Kotlin

```kotlin
fun isAnagram(s1: String, s2: String): Boolean {
    if (s1.length != s2.length) return false
    val freq = mutableMapOf<Char, Int>()
    for (ch in s1) freq[ch] = freq.getOrDefault(ch, 0) + 1
    for (ch in s2) {
        val count = freq.getOrDefault(ch, 0)
        if (count == 0) return false
        freq[ch] = count - 1
    }
    return true
}
```

## Pattern 2 — Grouping

### Python

In [ ]:
from collections import defaultdict

# Group anagrams — O(n · k log k) where k = average word length
def group_anagrams(words: list[str]) -> list[list[str]]:
    groups: dict[str, list[str]] = defaultdict(list)
    for word in words:
        key = "".join(sorted(word))    # canonical form — same for all anagrams
        groups[key].append(word)
    return list(groups.values())

print(group_anagrams(["eat", "tea", "tan", "ate", "nat", "bat"]))
# [['eat', 'tea', 'ate'], ['tan', 'nat'], ['bat']]

### Kotlin

```kotlin
fun groupAnagrams(words: List<String>): List<List<String>> {
    val groups = mutableMapOf<String, MutableList<String>>()
    for (word in words) {
        val key = word.toCharArray().sorted().joinToString("")
        groups.getOrPut(key) { mutableListOf() }.add(word)
    }
    return groups.values.toList()
}
```

## Pattern 3 — Two-Pass Complement Lookup

### Python

In [ ]:
# Two-sum — find indices of two numbers that sum to target  O(n)
def two_sum(nums: list[int], target: int) -> tuple[int, int]:
    seen = {}                           # val → index
    for i, val in enumerate(nums):
        complement = target - val
        if complement in seen:          # O(1) lookup
            return (seen[complement], i)
        seen[val] = i
    return (-1, -1)

print(two_sum([2, 7, 11, 15], 9))      # (0, 1)
print(two_sum([3, 2, 4], 6))           # (1, 2)

# Contains duplicate — O(n)
def has_duplicate(nums: list[int]) -> bool:
    return len(nums) != len(set(nums))

print(has_duplicate([1, 2, 3, 1]))     # True
print(has_duplicate([1, 2, 3, 4]))     # False

### Kotlin

```kotlin
fun twoSum(nums: IntArray, target: Int): Pair<Int, Int> {
    val seen = mutableMapOf<Int, Int>()   // val → index
    for ((i, value) in nums.withIndex()) {
        val complement = target - value
        if (complement in seen) return Pair(seen[complement]!!, i)
        seen[value] = i
    }
    return Pair(-1, -1)
}

fun hasDuplicate(nums: IntArray): Boolean = nums.size != nums.toHashSet().size
```

## Pattern 4 — Sliding Window with a Frequency Map

### Python

In [ ]:
# Longest substring without repeating characters — O(n)
def length_of_longest_substring(s: str) -> int:
    last_seen = {}    # char → last index seen
    left, best = 0, 0

    for right, ch in enumerate(s):
        if ch in last_seen and last_seen[ch] >= left:
            left = last_seen[ch] + 1    # shrink window past the duplicate
        last_seen[ch] = right
        best = max(best, right - left + 1)

    return best

print(length_of_longest_substring("abcabcbb"))   # 3  ("abc")
print(length_of_longest_substring("pwwkew"))     # 3  ("wke")
print(length_of_longest_substring(""))           # 0

### Kotlin

```kotlin
fun lengthOfLongestSubstring(s: String): Int {
    val lastSeen = mutableMapOf<Char, Int>()
    var left = 0; var best = 0
    for ((right, ch) in s.withIndex()) {
        val prev = lastSeen[ch]
        if (prev != null && prev >= left) left = prev + 1
        lastSeen[ch] = right
        best = maxOf(best, right - left + 1)
    }
    return best
}
```

## What Can Be a Key?

A key must be **hashable** — its hash value must be stable and consistent with equality.

| Type | Hashable? | Notes |
|---|---|---|
| `int`, `float`, `bool` | ✅ | Primitive types |
| `str` | ✅ | Immutable |
| `tuple` | ✅ | If all elements are hashable |
| `frozenset` | ✅ | Immutable set |
| `list` | ❌ | Mutable — hash would change |
| `dict` | ❌ | Mutable |
| `set` | ❌ | Mutable (use `frozenset` instead) |

In Kotlin, any object that correctly implements `hashCode()` and `equals()` can be a map key. Data classes do this automatically.

## HashMap vs TreeMap vs Array

| | `dict` / `HashMap` | `SortedDict` / `TreeMap` | Array / list |
|---|---|---|---|
| Lookup | O(1) avg | O(log n) | O(1) by index, O(n) by value |
| Insert | O(1) avg | O(log n) | O(1) end, O(n) middle |
| Keys ordered? | ❌ (insertion order in Python 3.7+) | ✅ sorted | N/A |
| Key type | Any hashable | Must be comparable | Integer index |
| Memory | Higher | Higher | Lowest |

Use a `TreeMap` (Python: `sortedcontainers.SortedDict`) when you need both O(log n) lookup *and* ordered iteration — e.g., range queries, finding the next larger key.

## Key Takeaways

- A hash map converts a key to a **bucket index** via a hash function — `index = hash(key) % size`. That arithmetic is why lookup is O(1).
- **Collisions** (two keys → same index) are resolved by chaining. With a good hash function and load factor, chains stay length ~1.
- All core operations — get, put, delete, contains — are **O(1) average, O(n) worst case**. Treat them as O(1) in practice.
- A **hash set** is a hash map without values — O(1) membership test, deduplification, and set algebra.
- **Keys must be hashable** — immutable types only. Lists and dicts cannot be keys; tuples and frozensets can.
- The four canonical hash map patterns: **frequency counting** (`Counter`), **grouping** (`defaultdict(list)`), **complement lookup** (two-sum), **sliding window** (last-seen index map).
- Use a **TreeMap** when you need sorted key order; use a plain dict/HashMap when you just need fast lookup.